In [1]:
# Install Pytorch & other libraries
%pip install "torch==2.4.1" tensorboard 
%pip install flash-attn "setuptools<71.0.0" scikit-learn 
 
# Install Hugging Face libraries
%pip install  --upgrade \
  "datasets==3.1.0" \
  "accelerate==1.2.1" \
  "hf-transfer==0.1.8"
  #"transformers==4.47.1" \
 
# ModernBERT is not yet available in an official release, so we need to install it from github
%pip install "git+https://github.com/huggingface/transformers.git@6e0515e99c39444caae39472ee1b2fd76ece32f1" --upgrade

  Obtaining dependency information for torch==2.4.1 from https://files.pythonhosted.org/packages/cc/df/5204a13a7a973c23c7ade615bafb1a3112b5d0ec258d8390f078fa4ab0f7/torch-2.4.1-cp312-cp312-manylinux1_x86_64.whl.metadata
  Obtaining dependency information for tensorboard from https://files.pythonhosted.org/packages/5d/12/4f70e8e2ba0dbe72ea978429d8530b0333f0ed2140cc571a48802878ef99/tensorboard-2.19.0-py3-none-any.whl.metadata
  Obtaining dependency information for sympy from https://files.pythonhosted.org/packages/99/ff/c87e0622b1dadea79d2fb0b25ade9ed98954c9033722eb707053d310d4f3/sympy-1.13.3-py3-none-any.whl.metadata
  Obtaining dependency information for networkx from https://files.pythonhosted.org/packages/b9/54/dd730b32ea14ea797530a4479b2ed46a6fb250f682a9cfb997e968bf0261/networkx-3.4.2-py3-none-any.whl.metadata
  Obtaining dependency information for nvidia-cuda-nvrtc-cu12==12.1.105 from https://files.pythonhosted.org/packages/b6/9f/c64c03f49d6fbc56196664d05dba14e3a561038a81a638eeb47f4

In [1]:
from datasets import load_dataset
 
# Dataset id from huggingface.co/dataset
dataset_id = "youralien/feedback_qesconv_16wayclassification"
 
# Load raw dataset
raw_dataset = load_dataset(dataset_id, split="train") # happens to be called train
 
print(f"Raw dataset size: {len(raw_dataset)}") 

/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Raw dataset size: 8179


In [2]:
split_dataset = raw_dataset.train_test_split(test_size=0.1)
print(f"Train dataset size: {len(split_dataset['train'])}")
print(f"Test dataset size: {len(split_dataset['test'])}")
split_dataset['train'][0]

Train dataset size: 7361
Test dataset size: 818


{'conv_index': 285,
 'helper_index': 12,
 'input': ['Helper: Earning a license for loans or for real estate sounds like a great way to add to your skillset. No, I do not have the premium package',
  "Seeker: I didn't want to pay for it either especially I did the trial before and didn't make too much of a difference",
  "Helper: It's understandable not to want to pay for something that didn't seem to pay off. What are some other options you've considered to support your job search?"],
 'Reflections-goodareas': 0,
 'Validation-goodareas': 0,
 'Empathy-goodareas': 0,
 'Questions-goodareas': 1,
 'Suggestions-goodareas': 0,
 'Self-disclosure-goodareas': 0,
 'Structure-goodareas': 0,
 'Professionalism-goodareas': 0,
 'Reflections-badareas': 0,
 'Validation-badareas': 0,
 'Empathy-badareas': 0,
 'Questions-badareas': 0,
 'Suggestions-badareas': 0,
 'Self-disclosure-badareas': 0,
 'Structure-badareas': 0,
 'Professionalism-badareas': 0}

In [3]:
def prepare_input_text(example):
    # Convert the last two items of input list to a single text
    return {
        'text': "\n".join(example['input'][-2:]),
        **{k:v for k,v in example.items() if k != 'input'}  # Keep other fields
    }

# Apply the preprocessing
split_dataset = split_dataset.map(prepare_input_text)
split_dataset['train'][0]

Map: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 818/818 [00:00<00:00, 5421.13 examples/s]


{'conv_index': 285,
 'helper_index': 12,
 'input': ['Helper: Earning a license for loans or for real estate sounds like a great way to add to your skillset. No, I do not have the premium package',
  "Seeker: I didn't want to pay for it either especially I did the trial before and didn't make too much of a difference",
  "Helper: It's understandable not to want to pay for something that didn't seem to pay off. What are some other options you've considered to support your job search?"],
 'Reflections-goodareas': 0,
 'Validation-goodareas': 0,
 'Empathy-goodareas': 0,
 'Questions-goodareas': 1,
 'Suggestions-goodareas': 0,
 'Self-disclosure-goodareas': 0,
 'Structure-goodareas': 0,
 'Professionalism-goodareas': 0,
 'Reflections-badareas': 0,
 'Validation-badareas': 0,
 'Empathy-badareas': 0,
 'Questions-badareas': 0,
 'Suggestions-badareas': 0,
 'Self-disclosure-badareas': 0,
 'Structure-badareas': 0,
 'Professionalism-badareas': 0,
 'text': "Seeker: I didn't want to pay for it either esp

In [4]:
from transformers import AutoTokenizer
 
# Model id to load the tokenizer
model_id = "answerdotai/ModernBERT-base"
# model_id = "google-bert/bert-base-uncased"

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.model_max_length = 512 # set model_max_length to 512 as prompts are not longer than 1024 tokens
 
# Tokenize helper 
 
# Tokenize helper function
def tokenize(batch):
    # return tokenizer(batch['text'], padding=True, truncation=True, return_tensors="pt")
    return tokenizer(batch['text'], padding='max_length', truncation=True, return_tensors="pt")


which_class = "Empathy-goodareas"
SKILL_OPTIONS = ["Reflections", "Validation", "Empathy", "Questions", "Suggestions", "Self-disclosure", "Structure", "Professionalism"]
goodareas_to_ignore = [f"{skill}-goodareas" for skill in SKILL_OPTIONS if f"{skill}-goodareas" != which_class]
badareas_to_ignore = [f"{skill}-badareas" for skill in SKILL_OPTIONS if f"{skill}-badareas" != which_class]
cols_to_remove = ['conv_index', 'helper_index', 'input', 'text']
cols_to_remove.extend(goodareas_to_ignore)
cols_to_remove.extend(badareas_to_ignore)
if which_class in split_dataset["train"].features.keys():
    split_dataset =  split_dataset.rename_column(which_class, "labels") # to match Trainer
tokenized_dataset = split_dataset.map(tokenize, batched=True, remove_columns=cols_to_remove)
 
tokenized_dataset["train"].features.keys()
# dict_keys(['labels', 'input_ids', 'attention_mask'])

Map: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 818/818 [00:00<00:00, 5742.92 examples/s]


dict_keys(['labels', 'input_ids', 'attention_mask'])

In [5]:
from transformers import AutoModelForSequenceClassification
 
# Prepare model labels - useful for inference
labels = ["not selected", "selected"]
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label
 
# Download the model from huggingface.co/models
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=num_labels, label2id=label2id, id2label=id2label,
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
You are attempting to use Flash Attention 2.0 without specifying a torch dtype. This might lead to unexpected behaviour
You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.
Flash Attention 2.0 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertForSequenceClassification is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `torch_dtype` argument. Example: `model = AutoModel.from_pretrain

In [6]:
import numpy as np
from sklearn.metrics import f1_score
 
# Metric helper method
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    score = f1_score(
            labels, predictions, labels=labels, pos_label=1, average="weighted"
        )
    return {"f1": float(score) if score == 1 else score}

In [7]:
from huggingface_hub import HfFolder
from transformers import Trainer, TrainingArguments
 
# Define training args
training_args = TrainingArguments(
    output_dir= f"ModernBERT-{which_class}-classifier",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    num_train_epochs=5,
    bf16=True, # bfloat16 training 
    optim="adamw_torch_fused", # improved optimizer 
    # logging & evaluation strategies
    logging_strategy="steps",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    # use_mps_device=True, # mps device is a mac thing
    metric_for_best_model="f1",
    # push to hub parameters
    report_to="tensorboard",
    push_to_hub=True,
    hub_strategy="every_save",
    hub_token=HfFolder.get_token(),
)
 
# Create a Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)
trainer.train()
# {'train_runtime': 3642.7783, 'train_samples_per_second': 1.235, 'train_steps_per_second': 0.04, 'train_loss': 0.535627057634551, 'epoch': 5.0}


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this 

Epoch,Training Loss,Validation Loss,F1
1,0.531900,0.499643,0.747323
2,0.431200,0.462252,0.786143
3,0.242300,0.588078,0.776529
4,0.042500,1.331632,0.769990
5,0.002100,1.596891,0.772265


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


TrainOutput(global_step=1155, training_loss=0.25711006651838103, metrics={'train_runtime': 315.6154, 'train_samples_per_second': 116.613, 'train_steps_per_second': 3.66, 'total_flos': 1.254159252261888e+16, 'train_loss': 0.25711006651838103, 'epoch': 5.0})

In [8]:
# Save processor and create model card
tokenizer.save_pretrained(f"ModernBERT-{which_class}-classifier")
trainer.create_model_card()
trainer.push_to_hub()

model.safetensors: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 598M/598M [00:27<00:00, 22.0MB/s]


CommitInfo(commit_url='https://huggingface.co/youralien/ModernBERT-Empathy-goodareas-classifier/commit/b8b8be909f71efa5b5c3bce7e4d6001093c98b20', commit_message='End of training', commit_description='', oid='b8b8be909f71efa5b5c3bce7e4d6001093c98b20', pr_url=None, repo_url=RepoUrl('https://huggingface.co/youralien/ModernBERT-Empathy-goodareas-classifier', endpoint='https://huggingface.co', repo_type='model', repo_id='youralien/ModernBERT-Empathy-goodareas-classifier'), pr_revision=None, pr_num=None)

In [51]:
import pandas as pd

# condition = "control"
condition = "treatment"
input_data = pd.read_csv(f"../Empathy-Mental-Health/dataset/all_{condition}_seekerhelper_pairs.csv")
input_data.head()

,id,seeker_post,response_post
0,aa6849fd46e7429b90ecff9bf8f7d388_1,Hi. Thanks. I don't even know where to start. ...,I understand the holidays were hard for you an...
1,aa6849fd46e7429b90ecff9bf8f7d388_2,It's just... everything feels like it’s fallin...,You are feeling that your parents have abandon...
2,aa6849fd46e7429b90ecff9bf8f7d388_3,It’s like they’ve completely erased me from th...,why do you feel that reaching out wouldn't work?
3,aa6849fd46e7429b90ecff9bf8f7d388_4,"Because they've ignored me for so long, and ev...",Is it anything in specific you feel they don't...
4,aa6849fd46e7429b90ecff9bf8f7d388_5,They don’t understand why I’m upset. It’s like...,"I see, you not only feel abandoned by them but..."


In [34]:
from transformers import pipeline
 
# load model from huggingface.co/models using our repository id
classifier = pipeline("sentiment-analysis", model=f"ModernBERT-{which_class}-classifier", device=0)

sample = f"Seeker: {input_data.loc[0, "seeker_post"]}\nHelper: {input_data.loc[0, "response_post"]}"
pred = classifier(sample)
print(pred)

def binary_prediction_seeker_response_post(seeker, helper):
    sample = f"Seeker: {seeker}\nHelper: {helper}"
    pred = classifier(sample)
    return int(pred[0]['label'] == 'selected')

Device set to use cuda:0


[{'label': 'selected', 'score': 0.7994047999382019}]


In [52]:
# output_preds = input_data.apply(binary_prediction_seeker_response_post, axis=0)

strengths = [binary_prediction_seeker_response_post(input_data.loc[i, "seeker_post"], input_data.loc[i, "response_post"])
             for i in range(len(input_data))]

In [53]:
input_data[f"{which_class}"] = strengths

In [54]:
input_data.head()

,id,seeker_post,response_post,Empathy-goodareas
0,aa6849fd46e7429b90ecff9bf8f7d388_1,Hi. Thanks. I don't even know where to start. ...,I understand the holidays were hard for you an...,1
1,aa6849fd46e7429b90ecff9bf8f7d388_2,It's just... everything feels like it’s fallin...,You are feeling that your parents have abandon...,1
2,aa6849fd46e7429b90ecff9bf8f7d388_3,It’s like they’ve completely erased me from th...,why do you feel that reaching out wouldn't work?,0
3,aa6849fd46e7429b90ecff9bf8f7d388_4,"Because they've ignored me for so long, and ev...",Is it anything in specific you feel they don't...,0
4,aa6849fd46e7429b90ecff9bf8f7d388_5,They don’t understand why I’m upset. It’s like...,"I see, you not only feel abandoned by them but...",0


In [55]:
input_data.to_csv(f'all_{condition}_seekerhelper_pairs_{which_class}.csv')